# 📚 File Search (RAG) Agent — Solution

This notebook provides a complete working solution to the File Search challenge.  
The agent answers questions about AtlasTech Maroc's internal knowledge base —
data the model could not possibly know without the uploaded document.

In [1]:
# ! pip install -r ../../../Installation/requirements.txt -U

In [2]:
import os

from azure.identity.aio import AzureCliCredential
from azure.ai.agents.aio import AgentsClient
from dotenv import load_dotenv

from agent_framework.azure import AzureAIClient, AzureAIProjectAgentProvider

In [3]:
load_dotenv("../../../.env")

True

In [4]:
AgentName = "AtlasTech-KB-Agent"

# SOLUTION — Task 1: Agent Instructions
AgentInstructions = """You are an internal knowledge assistant for AtlasTech Maroc,
an enterprise software company headquartered in Casablanca.

Your role is to help employees find accurate information from the company's
internal knowledge base, covering products, pricing, HR policies, and IT procedures.

Rules:
- Always use the File Search tool to retrieve relevant content before answering.
- Answer ONLY using information found in the uploaded company documents.
- Quote exact figures (prices, SKUs, deadlines, allowances) as they appear in the document.
- If the answer is not in the documents, say clearly:
  "I could not find that information in the AtlasTech knowledge base."
- Never invent product codes, prices, contact details, or policy rules."""

In [ ]:
file = None
vector_store = None

async with (
    AzureCliCredential() as credential,
    AgentsClient(
        endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
        credential=credential
    ) as agents_client,
    AzureAIProjectAgentProvider(credential=credential) as provider,
):
    try:
        file_path = "../files/atlastech_kb.md"
        print(f"Uploading file from: {file_path}")

        # SOLUTION — Task 2: Upload file and create vector store
        file = await agents_client.files.upload_and_poll(
            file_path=file_path, purpose="assistants"
        )
        print(f"Uploaded file, file ID: {file.id}")

        vector_store = await agents_client.vector_stores.create_and_poll(
            file_ids=[file.id], name="atlastech_kb"
        )
        print(f"Created vector store, vector store ID: {vector_store.id}")

        # SOLUTION — Task 3: Configure the file search tool
        client = AzureAIClient(project_client=agents_client)
        file_search_tool = client.get_file_search_tool(vector_store_ids=[vector_store.id])

        # SOLUTION — Task 4: Create the agent, run a query, print the result
        agent = await provider.create_agent(
            name=AgentName,
            instructions=AgentInstructions,
            tools=file_search_tool,
        )

        query = "What is the price and SKU for the 5-seat AtlasCloud licence, and what support tier gives me a 1-hour emergency response?"
        print(f"\n# User: '{query}'")
        response = await agent.run(query)
        print(f"# Agent: {response.text}")

    finally:
        if vector_store:
            await agents_client.vector_stores.delete(vector_store.id)
            print(f"\nCleaned up vector store: {vector_store.id}")
        if file:
            await agents_client.files.delete(file.id)
            print(f"Cleaned up file: {file.id}")

Uploading file from: ../files/atlastech_kb.md
Uploaded file, file ID: assistant-NzrBi5csNjkLVqCQD2tVdD
Created vector store, vector store ID: vs_brMhaubIrfZXHFOmU2qky6bs


AzureAIClient does not support runtime tools or structured_output overrides after agent creation. Use AzureOpenAIResponsesClient instead.



# User: 'What is the price and SKU for the 5-seat AtlasCloud licence, and what support tier gives me a 1-hour emergency response?'
